# Shank Geometry and Buckling Safety for a Penetrating Neural Probe

Estimates whether a probe shank can be inserted into tissue without buckling. Uses Euler column buckling for a
rectangular shank and compares the critical load with an estimated insertion force (tip puncture + friction along the shank).

**How to use:** Runtime > Run all. Change the inputs in the "Inputs" cell and run again.

**Important:** the insertion-force parameters are illustrative placeholders. Replace them with values from
published insertion-force measurements before using any number in a paper. Euler buckling is a first-order estimate;
check the final design with FEM (for example FEniCS or COMSOL).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Inputs

In [ ]:
MATERIAL = "Si"                      # Si, SiC, Parylene-C, Polyimide
E_GPA = {"Si": 169.0, "SiC": 400.0, "Parylene-C": 4.0, "Polyimide": 2.5}   # Young's modulus, GPa

WIDTH_UM = 50.0                      # shank width (um)
THICKNESS_UM_LIST = [10, 15, 20, 30, 50]    # shank thickness values to compare (um)
LENGTH_MM_RANGE = (1.0, 15.0)        # shank length range (mm)
INSERTION_DEPTH_MM = 5.0             # depth of insertion into tissue (mm)

K_FACTOR = 2.0                       # effective length factor: 2.0 fixed-free (conservative), 0.7 fixed-pinned
REQUIRED_SAFETY_FACTOR = 3.0

# Insertion force model (PLACEHOLDERS - replace with measured / published values)
PUNCTURE_PRESSURE_PA = 1.0e6         # force per unit tip cross-section needed to puncture tissue (Pa)
FRICTION_SHEAR_PA = 1.0e3            # shear stress along the shank walls (Pa)

## Model

In [ ]:
def euler_critical_load_mN(width_um, thickness_um, length_mm, material=MATERIAL, k=K_FACTOR):
    # P_cr = pi^2 * E * I / (K*L)^2, bending about the weak axis: I = w * t^3 / 12
    E = E_GPA[material] * 1e9
    w = width_um * 1e-6
    t = np.asarray(thickness_um, dtype=float) * 1e-6
    L = np.asarray(length_mm, dtype=float) * 1e-3
    I = w * t ** 3 / 12.0
    return np.pi ** 2 * E * I / (k * L) ** 2 * 1e3      # mN

def insertion_force_mN(width_um, thickness_um, depth_mm=INSERTION_DEPTH_MM):
    w = width_um * 1e-6
    t = np.asarray(thickness_um, dtype=float) * 1e-6
    tip = PUNCTURE_PRESSURE_PA * w * t
    friction = FRICTION_SHEAR_PA * 2.0 * (w + t) * depth_mm * 1e-3
    return (tip + friction) * 1e3                       # mN

def safety_factor(width_um, thickness_um, length_mm, material=MATERIAL):
    return euler_critical_load_mN(width_um, thickness_um, length_mm, material) / insertion_force_mN(width_um, thickness_um)

def max_length_mm(width_um, thickness_um, sf=REQUIRED_SAFETY_FACTOR, material=MATERIAL, k=K_FACTOR):
    # solve P_cr(L) = sf * F_ins for L
    E = E_GPA[material] * 1e9
    w = width_um * 1e-6
    t = thickness_um * 1e-6
    I = w * t ** 3 / 12.0
    f_req = sf * insertion_force_mN(width_um, thickness_um) * 1e-3     # N
    return np.sqrt(np.pi ** 2 * E * I / f_req) / k * 1e3

## Critical load vs length

In [ ]:
L = np.linspace(LENGTH_MM_RANGE[0], LENGTH_MM_RANGE[1], 200)
plt.figure(figsize=(7, 5))
for t in THICKNESS_UM_LIST:
    plt.semilogy(L, euler_critical_load_mN(WIDTH_UM, t, L), label=f"t = {t} um")
    plt.axhline(insertion_force_mN(WIDTH_UM, t), ls=":", color="gray", alpha=0.6)
plt.xlabel("Shank length (mm)"); plt.ylabel("Critical buckling load (mN)")
plt.title(f"{MATERIAL}, width {WIDTH_UM:.0f} um (dotted: estimated insertion force)")
plt.legend(); plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("buckling_load.png", dpi=200)
plt.show()

## Safety factor map (thickness x length)

In [ ]:
t_grid = np.linspace(8, 60, 120)
l_grid = np.linspace(LENGTH_MM_RANGE[0], LENGTH_MM_RANGE[1], 120)
TT, LL = np.meshgrid(t_grid, l_grid)
SF = safety_factor(WIDTH_UM, TT, LL)

plt.figure(figsize=(7, 5))
im = plt.pcolormesh(TT, LL, np.log10(SF), shading="auto", cmap="viridis")
cs = plt.contour(TT, LL, SF, levels=[REQUIRED_SAFETY_FACTOR], colors="white", linewidths=2)
plt.clabel(cs, fmt=lambda v: f"SF = {v:.0f}")
plt.colorbar(im, label="log10(safety factor)")
plt.xlabel("Shank thickness (um)"); plt.ylabel("Shank length (mm)")
plt.title(f"{MATERIAL}, width {WIDTH_UM:.0f} um")
plt.tight_layout()
plt.savefig("safety_map.png", dpi=200)
plt.show()

## Design table: maximum shank length for the required safety factor

In [ ]:
rows = []
for m in E_GPA:
    for t in THICKNESS_UM_LIST:
        rows.append({
            "material": m,
            "thickness_um": t,
            "width_um": WIDTH_UM,
            "insertion force (mN)": round(float(insertion_force_mN(WIDTH_UM, t)), 3),
            f"max length for SF={REQUIRED_SAFETY_FACTOR:.0f} (mm)": round(float(max_length_mm(WIDTH_UM, t, material=m)), 2),
            "cross-section (um^2)": WIDTH_UM * t,
        })
df = pd.DataFrame(rows)
df.to_csv("max_shank_length.csv", index=False)
print(df.to_string(index=False))

## Next steps for the paper
1. Replace the placeholder insertion-force parameters with published measurements for your target tissue and insertion speed.
2. Validate the Euler estimate with a FEM buckling analysis for the final geometry (tapered tip, tissue support).
3. Combine with the impedance/noise notebook: smaller sites and thinner shanks reduce tissue damage but raise impedance and noise.